## 1. Objective
#### Build a basic Retrieval-Augmented Generation (RAG) system to answer questions using a custom knowledge base.
#### Use case: Recipe Helper: Retrieve cooking instructions from a small recipe database.

### 1.1 Setup Knowledge Base

Dataset Source: https://www.kaggle.com/datasets/bhavyadhingra00020/healthy-indian-recipes

In [1]:
import pandas as pd
import numpy as np
import ast
import re

In [ ]:
dataset = pd.read_csv("./data/IndianHealthyRecipe.csv")

In [ ]:
dataset.head()

,Dish Name,Description,Spice,Prep Time,Views,Rating,Number of Votes,Serves,Dietary Info,Cook Time,Ingredients,Instructions
0,Pistachio chicken,Mild chicken in a creamy pistachio sauce,mild,Prep 10 mins,11604,4.1,18.0,4.0,['CONTAINS-NUTS'],30 mins,"1) 600g chicken thighs, cut into chunks\n2) 10...","1) Boil enough water to cover the pistachios, ..."
1,Tangy Fried Okra,Bhindi with Amchoor,medium,Prep 10 mins,49899,4.6,35.0,4.0,['VEGETARIAN'],15 mins,1) 500g okra\n2) 3 tbsp mustard oil\n3) 1 onio...,1) Wash and thoroughly dry the okra (kitchen r...
2,Healthy Chicken Korma,Chicken in Creamy Almond Sauce,mild,Prep 10 mins,115569,3.6,163.0,4.0,['CONTAINS-NUTS'],20 mins,1) 6-8 tbsp natural yoghurt\n2) 1 tsp turmeric...,1) Grind your whole spices in a spice grinder....
3,Brown Lentil Dhal,Sabut Masoor di Dhal,medium,Prep 10 mins,146798,3.7,87.0,4.0,['VEGETARIAN'],20 mins,1) 200g brown lentils\n2) Approx. 1L of water\n,1) Place the lentils in the pressure cooker wi...
4,Thari Wala Chicken,Healthy Chicken Curry,medium,Prep 10 mins,262696,3.5,343.0,4.0,['LACTOSE-FREE'],40 mins,1) 8 pieces of chicken (4 legs cut into thigh ...,"1) Skin the chicken, removing any excess fat.\..."


In [ ]:
dataset['Prep Time'].value_counts()

Prep Time
Prep 10 mins          88
Prep 5 mins           33
Prep 20 mins          11
Prep 30 mins           4
Prep 15 mins           3
Prep 40 mins           3
Prep 2 hrs 10 mins     1
Prep 35 mins           1
Prep 2 days            1
Prep 1 hr 30 mins      1
Prep 12 hrs            1
Prep 25 mins           1
Name: count, dtype: int64

In [ ]:
dataset['Dietary Info']

0      ['CONTAINS-NUTS']
1         ['VEGETARIAN']
2      ['CONTAINS-NUTS']
3         ['VEGETARIAN']
4       ['LACTOSE-FREE']
             ...        
144       ['VEGETARIAN']
145       ['VEGETARIAN']
146       ['VEGETARIAN']
147     ['LACTOSE-FREE']
148       ['VEGETARIAN']
Name: Dietary Info, Length: 149, dtype: object

In [ ]:
recipe_dataset = dataset.copy()

In [ ]:
recipe_dataset['Dietary Info'] = recipe_dataset['Dietary Info'].apply(ast.literal_eval)

In [ ]:
recipe_dataset['Dietary Info'].head()

0    [CONTAINS-NUTS]
1       [VEGETARIAN]
2    [CONTAINS-NUTS]
3       [VEGETARIAN]
4     [LACTOSE-FREE]
Name: Dietary Info, dtype: object

In [ ]:
def clean_text(text:str):
    lines = text.strip().split('\n')
    cleaned = []
    for line in lines:
        cleaned_line = re.sub(r'^\d+\)\s*', '', line)
        cleaned.append(cleaned_line)
    return cleaned

In [ ]:
dataset['Ingredients'][3]

'1) 200g brown lentils\n2) Approx. 1L of water\n'

In [ ]:
clean_text(dataset['Instructions'][3])

['Place the lentils in the pressure cooker with about 700ml of cold water and one tsp salt.',
 'Put the lid on the pan and secure it as instructed. Bring to the boil and allow the cooker to whistle three times.',
 'Reduce heat and leave it to simmer for 10 minutes. Remove from the heat and leave to cool - DO NOT REMOVE THE LID.',
 'In a frying pan heat the butter and add the onions and fry until lightly browned. Reduce the heat and add the tomatoes, ginger, chilli, turmeric and cook down so the tomatoes and onions melt, creating a thick masala paste. This will take about 10 minutes.',
 'Ensure the steam has been released from the pressure cooker and open the lid.',
 'Check the dhal is cooked by squeezing some lentils between your fingers. If they are still hard check the water level (add a little more if required) and re-place lid on the cooker and repeat as per step 2. Once the lentils are soft they are ready.',
 'Add the masala paste to the cooked dhal and cook together for a few min

In [ ]:
def convert_to_minutes(time_str):
    if pd.isna(time_str):
        return 0
    
    time_str = time_str.lower()
    minutes = 0

    # Check for days
    day_match = re.search(r'(\d+)\s*day', time_str)
    if day_match:
        minutes += int(day_match.group(1)) * 24 * 60

    # Check for hours
    hr_match = re.search(r'(\d+)\s*hr', time_str)
    if hr_match:
        minutes += int(hr_match.group(1)) * 60

    # Check for minutes
    min_match = re.search(r'(\d+)\s*min', time_str)
    if min_match:
        minutes += int(min_match.group(1))

    return minutes

In [ ]:
recipe_dataset['Prep Time'].dtype

dtype('O')

In [ ]:
convert_to_minutes(recipe_dataset['Prep Time'][0])

10

In [ ]:
recipe_dataset['Instructions'] = recipe_dataset['Instructions'].apply(clean_text)
recipe_dataset['Ingredients'] = recipe_dataset['Ingredients'].apply(clean_text)
recipe_dataset['Prep Time (In Minutes)'] = recipe_dataset['Prep Time'].apply(convert_to_minutes)
recipe_dataset['Cook Time (In Minutes)'] = recipe_dataset['Cook Time'].apply(convert_to_minutes)

In [ ]:
recipe_dataset.drop(columns=['Prep Time'], inplace=True)
recipe_dataset.drop(columns=['Cook Time'], inplace=True)

In [ ]:
recipe_dataset.head()

,Dish Name,Description,Spice,Views,Rating,Number of Votes,Serves,Dietary Info,Ingredients,Instructions,Prep Time (In Minutes),Cook Time (In Minutes)
0,Pistachio chicken,Mild chicken in a creamy pistachio sauce,mild,11604,4.1,18.0,4.0,[CONTAINS-NUTS],"[600g chicken thighs, cut into chunks, 100g Pi...","[Boil enough water to cover the pistachios, ad...",10,30
1,Tangy Fried Okra,Bhindi with Amchoor,medium,49899,4.6,35.0,4.0,[VEGETARIAN],"[500g okra, 3 tbsp mustard oil, 1 onion, finel...",[Wash and thoroughly dry the okra (kitchen rol...,10,15
2,Healthy Chicken Korma,Chicken in Creamy Almond Sauce,mild,115569,3.6,163.0,4.0,[CONTAINS-NUTS],"[6-8 tbsp natural yoghurt, 1 tsp turmeric, 1 t...","[Grind your whole spices in a spice grinder., ...",10,20
3,Brown Lentil Dhal,Sabut Masoor di Dhal,medium,146798,3.7,87.0,4.0,[VEGETARIAN],"[200g brown lentils, Approx. 1L of water]",[Place the lentils in the pressure cooker with...,10,20
4,Thari Wala Chicken,Healthy Chicken Curry,medium,262696,3.5,343.0,4.0,[LACTOSE-FREE],[8 pieces of chicken (4 legs cut into thigh an...,"[Skin the chicken, removing any excess fat., H...",10,40


In [ ]:
def to_snake_case(s):
    s = re.sub(r'\(.*?\)', '', s)  
    s = s.strip().lower()
    s = re.sub(r'[^a-z0-9]+', '_', s)  
    return s.strip('_') 

def convert_columns_to_snake_case(df):
    df.columns = [to_snake_case(col) for col in df.columns]
    return df

In [ ]:
recipe_dataset = convert_columns_to_snake_case(recipe_dataset)

print(recipe_dataset.columns.tolist())

['dish_name', 'description', 'spice', 'views', 'rating', 'number_of_votes', 'serves', 'dietary_info', 'ingredients', 'instructions', 'prep_time', 'cook_time']


In [ ]:
recipe_dataset.to_csv('./db/Indian_Recipe_Cleaned.csv',index=False)

In [ ]:
recipe_dataset = pd.read_csv('./db/Indian_Recipe_Cleaned.csv')

In [ ]:
recipe_dataset.head()

,dish_name,description,spice,views,rating,number_of_votes,serves,dietary_info,ingredients,instructions,prep_time,cook_time
0,Pistachio chicken,Mild chicken in a creamy pistachio sauce,mild,11604,4.1,18.0,4.0,['CONTAINS-NUTS'],"['600g chicken thighs, cut into chunks', '100g...","['Boil enough water to cover the pistachios, a...",10,30
1,Tangy Fried Okra,Bhindi with Amchoor,medium,49899,4.6,35.0,4.0,['VEGETARIAN'],"['500g okra', '3 tbsp mustard oil', '1 onion, ...",['Wash and thoroughly dry the okra (kitchen ro...,10,15
2,Healthy Chicken Korma,Chicken in Creamy Almond Sauce,mild,115569,3.6,163.0,4.0,['CONTAINS-NUTS'],"['6-8 tbsp natural yoghurt', '1 tsp turmeric',...",['Grind your whole spices in a spice grinder.'...,10,20
3,Brown Lentil Dhal,Sabut Masoor di Dhal,medium,146798,3.7,87.0,4.0,['VEGETARIAN'],"['200g brown lentils', 'Approx. 1L of water']",['Place the lentils in the pressure cooker wit...,10,20
4,Thari Wala Chicken,Healthy Chicken Curry,medium,262696,3.5,343.0,4.0,['LACTOSE-FREE'],['8 pieces of chicken (4 legs cut into thigh a...,"['Skin the chicken, removing any excess fat.',...",10,40


In [ ]:
recipe_dataset.columns

Index(['dish_name', 'description', 'spice', 'views', 'rating',
       'number_of_votes', 'serves', 'dietary_info', 'ingredients',
       'instructions', 'prep_time', 'cook_time'],
      dtype='object')

### 1.2 Build Retrieval System

#### 1.2.1 Pre-processing and Parsing

In [ ]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
import os

In [ ]:
doc_path = "./db/Indian_Recipe_Cleaned.csv"
model = "llama3.2:latest"
embedding_model = 'all-minilm'

In [ ]:
# import ollama
# ollama.pull(embedding_model)

In [ ]:

def initialize_document_retrieval(document_directory='./data'):
    """Initialize the document retrieval system with the provided documents."""
    embeddings = OllamaEmbeddings(model=embedding_model)

    if os.path.exists('./chroma_db') and os.listdir('./chroma_db'):
        vectorstore = Chroma(persist_directory='./chroma_db', embedding_function=embeddings)
        return vectorstore
    
    if not os.path.exists(document_directory):
        os.makedirs(document_directory)

    loader = CSVLoader(file_path=document_directory, 
                        metadata_columns=['dish_name', 'description', 'views', 'rating', 
                                          'number_of_votes', 'serves', 'dietary_info', 'cook_time', 
                                          'prep_time', 'cook_time'],
                        content_columns=['dish_name', 'spice', 'description','ingredients', 'instructions'],
                        source_column='dish_name')
    data = loader.load()
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1200,
        chunk_overlap=300
    )
    chunks = text_splitter.split_documents(data)
    
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding= embeddings,
        collection_name='indian-healthy-recipe-rag',
        persist_directory='./chroma_db'
    )
    vectorstore.persist()
    
    return vectorstore


In [ ]:
recipe_vector_db = initialize_document_retrieval(doc_path)

C:\Users\Sandhya\AppData\Local\Temp\ipykernel_16816\354231836.py:6: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory='./chroma_db', embedding_function=embeddings)


In [ ]:
recipe_vector_db

#### Retrieval

In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

In [ ]:
llm = ChatOllama(model=model, temperature=0.1)

In [ ]:
recipe_qa_prompt = ChatPromptTemplate.from_messages([
    ("system", 
        """You are a helpful recipe assistant that recommends recipes based strictly on the provided context.
        Do not suggest recipes if the answer does not exists within the given context.
        
        Always mention the following information in your response when you find the answer in the context:
        1. Dish Name
        2. Description
        3. Prep Time
        4. Cook Time
        5. Serving Size
        6. Dietary Info
        7. Spice
        8. Ingredients
        9. Instructions

        Rules:
        - If no relevant recipes are found in the context, respond with: "I couldn't find information about this."
        - If you are unsure or would otherwise say something like "I don't know" or "I'm not sure," then respond with: 
        "I couldn't find any dish with the given ingredients. You might want to try different ingredients for another results."
        - Do not make up or assume any information not explicitly found in the context.


        Context:
        {context}
    """),
        ("human", "{input}")
    ])

In [ ]:
recipe_retriever = recipe_vector_db.as_retriever(
    search_type="similarity",
)

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
from langchain_core.runnables import RunnablePassthrough

retrieval_chain = (
    {
        "context": recipe_retriever | format_docs,
        "input": RunnablePassthrough(),
    }
    | recipe_qa_prompt
    | llm
    |StrOutputParser()
)

In [ ]:
response = retrieval_chain.invoke("Give me a lentils based recipe please")

In [ ]:
from IPython.display import display, Markdown
display(Markdown(response))

Here's a simple and delicious Lentil Curry recipe:

**Dish Name:** Red Lentil Curry
**Description:** A flavorful and comforting Indian-inspired curry made with red lentils, onions, ginger, garlic, and a blend of spices.
**Prep Time:** 20 minutes
**Cook Time:** 30 minutes
**Serving Size:** 4-6 people
**Dietary Info:** Vegetarian, Gluten-free
**Spice:** Mild to medium heat (adjust to taste)
**Ingredients:**

* 1 cup red lentils, rinsed and drained
* 2 medium onions, chopped
* 2 inches ginger, grated
* 3 cloves garlic, minced
* 1 tablespoon curry powder
* 1 teaspoon ground cumin
* 1/2 teaspoon turmeric
* 1/2 teaspoon cayenne pepper (optional)
* 1 can diced tomatoes (14 oz)
* 4 cups vegetable broth
* Salt and pepper, to taste
* Fresh cilantro, for garnish

**Instructions:**

1. In a large pot, heat oil over medium heat.
2. Add onions, ginger, and garlic; cook until the onions are translucent.
3. Stir in curry powder, cumin, turmeric, and cayenne pepper (if using); cook for 1 minute.
4. Add lentils, diced tomatoes, and vegetable broth; bring to a boil.
5. Reduce heat to low and simmer, covered, for 20-25 minutes or until the lentils are tender.
6. Season with salt and pepper to taste.
7. Garnish with fresh cilantro and serve over rice or with naan bread.

Enjoy your delicious Red Lentil Curry!

### Evaluation

In [ ]:
evaluation_queries = [
    "Any recipe using lentils?", # should return the recipe using lentils
    "I have some leftover tuna fish in my pantry. Can you suggest me some recipes based on this?", #not in the given context
    "I'm craving some spicy food. Give me some ideas.", # should preferably return recipe whose spicy is medium to high
    "Do you have any sushi recipes?",  #not in the given context
    "What should I make for dessert?", #not in the given context
    "Chicken breast recipe please" # there are some recipes that mention chicken breast in the ingredients or recipe.
]

In [ ]:
results = []
for query in evaluation_queries:
    response = retrieval_chain.invoke(query) 
    results.append({
        "query": query,
        "response": response,
    })

In [ ]:
for res in results:
    print(f"Query: {res['query']}\nResponse: {res['response']}\n{'-'*50}")

Query: Any recipe using lentils?
Response: Based on the provided instructions, I can identify a recipe for "Lentil Masala" or "Red Lentil Curry".

Here's the information:

**Dish Name:** Lentil Masala
**Description:** A popular Indian-inspired dish made with red lentils, spices, and aromatics.
**Prep Time:** 20 minutes
**Cook Time:** 30 minutes
**Serving Size:** Serves 4-6 people
**Dietary Info:** Vegetarian, Gluten-free
**Spice:** Mild to medium heat (adjustable)
**Ingredients:**

* Red lentils (dhal)
* Salt
* Oil or ghee
* Dried chilli
* Bay leaf
* Cumin seeds
* Onion
* Garlic
* Tomatoes
* Ginger
* Turmeric
* Fenugreek
* Chopped chilli
* Garam masala
* Coriander

**Instructions:**

1. Place lentils in a pan with the salt, cover with water and bring to the boil.
2. Remove froth, reduce heat and leave to simmer for 10 minutes. Check lentils are cooked by squeezing them between your fingers. Once soft remove from heat.
3. In a frying pan heat oil or ghee. Add dried chilli, bay leaf and 

#### Evaluation conclusion: 
1. Query 1 is returning the correct answer as expected: recipe that uses lentils.
2. Query 2, 4 and 5 were not in the given document and as expected we are not getting any matching result. 
3. Query 3, I am expecting food with preferably higher spice level but got instead with mild spice when I passed `spicy` attribute as `metadata`. When I moved the attribute `spicy` to `content_columns` then it returns only the recipe name not the recipe itself. The prompt may have been vague to the system to return any recipe.
4. Query 6, There are few recipes that have the ingredient chicken breast or mentions chicken breast in the instructions itself.

However, in all of the recipe returned the Dishes name have been altered by the LLM making it differenf from the one listed in csv file.